# 🔎 Punto 4 — Análisis Cualitativo de Fallos
## Examen Parcial: Redes Neuronales y Aprendizaje Profundo

Este notebook identifica y analiza sistemáticamente los **ejemplos mal clasificados** por ambos modelos,
cumpliendo el requisito de comentar al menos **5 casos** con hipótesis sobre sus causas.

| Análisis | Descripción |
|----------|-------------|
| **Fallos compartidos** | Errores que cometen *ambos* modelos → casos más difíciles |
| **Fallos exclusivos** | Errores que comete solo uno de los modelos |
| **Tipología de error** | FP vs FN, confianza del error, patrón visual |
| **Grad-CAM en fallos** | Activaciones en imágenes mal clasificadas |
| **Hipótesis** | Explicación razonada por cada caso |

> **Prerrequisito:** `tire_classification.ipynb` ejecutado  
> Modelos cargados desde `/content/TireCNN_final.pth` y `/content/EfficientNet_final.pth`

---
## 0. Instalación y configuración

In [ ]:
!pip install -q kagglehub torchvision matplotlib seaborn scikit-learn pandas grad-cam

In [ ]:
import os, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image, ImageStat

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE      = 224
DATA_DIR      = Path('/content/dataset_bruto/Tire Textures')
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print(f'Dispositivo: {DEVICE}')

---
## 1. Reconstrucción de modelos y split

In [ ]:
# ── Arquitecturas (idénticas al Punto 1) ─────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        if pool: layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class TireCNN(nn.Module):
    def __init__(self, num_classes=2, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,32), ConvBlock(32,64),
            ConvBlock(64,128), ConvBlock(128,256),
        )
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(256,128), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.global_avg_pool(self.features(x)))


# ── Dataset y split ───────────────────────────────────────────────────────────
raw_dataset = datasets.ImageFolder(root=str(DATA_DIR), transform=None)
CLASS_NAMES = raw_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
targets     = [s[1] for s in raw_dataset.samples]
indices     = list(range(len(raw_dataset)))

train_idx, temp_idx = train_test_split(indices, test_size=0.30,
                                        stratify=targets, random_state=SEED)
_, test_idx = train_test_split(temp_idx, test_size=0.50,
                                stratify=[targets[i] for i in temp_idx],
                                random_state=SEED)

print(f'Clases: {CLASS_NAMES}  |  Test: {len(test_idx)} imágenes')


# ── Cargar pesos ──────────────────────────────────────────────────────────────
def load_model(arch, path, num_classes, device):
    if arch == 'tirecnn':
        model = TireCNN(num_classes)
    else:
        model = models.efficientnet_b0(weights=None)
        in_f  = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.3, inplace=True), nn.Linear(in_f, 256),
            nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    ckpt  = torch.load(path, map_location=device)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state)
    model.eval().to(device)
    return model

model_scratch = load_model('tirecnn',    '/content/TireCNN_final.pth',      NUM_CLASSES, DEVICE)
model_effnet  = load_model('efficientnet','/content/EfficientNet_final.pth', NUM_CLASSES, DEVICE)
print('✅ Modelos cargados.')

---
## 2. Recolección exhaustiva de todos los errores en el conjunto de test

In [ ]:
@torch.no_grad()
def collect_failures(model, test_idx, raw_dataset, val_tf, device, model_name):
    """
    Recorre todo el conjunto de test y recopila los ejemplos mal clasificados
    con metadatos completos para análisis posterior.
    """
    failures = []
    model.eval()
    for idx in test_idx:
        img_path, true_label = raw_dataset.samples[idx]
        img_pil    = Image.open(img_path).convert('RGB')
        img_tensor = val_tf(img_pil).unsqueeze(0).to(device)

        logits = model(img_tensor)
        probs  = F.softmax(logits, dim=1).cpu().squeeze()
        pred   = probs.argmax().item()

        if pred != true_label:
            error_type = (
                'Falso Positivo (FP)'
                if pred == 1 and true_label == 0    # predicho dañado, era buena
                else 'Falso Negativo (FN)'
                if pred == 0 and true_label == 1    # predicho buena, era dañada
                else f'Error clase {true_label}→{pred}'
            )
            # Características visuales básicas
            stat    = ImageStat.Stat(img_pil)
            brightness  = np.mean(stat.mean)
            contrast    = np.mean(stat.stddev)
            img_arr     = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE)))
            blurriness  = cv_laplacian_var(img_arr)

            failures.append({
                'model':       model_name,
                'img_path':    img_path,
                'img_pil':     img_pil,
                'img_tensor':  val_tf(img_pil).unsqueeze(0),
                'true_label':  true_label,
                'true_name':   CLASS_NAMES[true_label],
                'pred_label':  pred,
                'pred_name':   CLASS_NAMES[pred],
                'confidence':  probs[pred].item(),
                'true_conf':   probs[true_label].item(),
                'error_type':  error_type,
                'brightness':  brightness,
                'contrast':    contrast,
                'blurriness':  blurriness,
                'probs':       probs.numpy(),
            })
    return failures


def cv_laplacian_var(img_arr):
    """Varianza del Laplaciano como proxy de nitidez (mayor = más nítida)."""
    try:
        import cv2
        gray = cv2.cvtColor(img_arr, cv2.COLOR_RGB2GRAY)
        return cv2.Laplacian(gray, cv2.CV_64F).var()
    except:
        gray = np.mean(img_arr, axis=2).astype(np.uint8)
        kernel = np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=np.float64)
        from scipy.ndimage import convolve
        lap = convolve(gray.astype(np.float64), kernel)
        return lap.var()


failures_scratch = collect_failures(model_scratch, test_idx, raw_dataset,
                                     val_tf, DEVICE, 'TireCNN')
failures_effnet  = collect_failures(model_effnet,  test_idx, raw_dataset,
                                     val_tf, DEVICE, 'EfficientNet-B0')

print(f'\nFallos TireCNN:        {len(failures_scratch)}')
print(f'Fallos EfficientNet-B0: {len(failures_effnet)}')

# FP vs FN
for name, fails in [('TireCNN', failures_scratch), ('EfficientNet-B0', failures_effnet)]:
    from collections import Counter
    etype = Counter(f['error_type'] for f in fails)
    print(f'  {name}: {dict(etype)}')

---
## 3. Identificación de fallos compartidos vs exclusivos

In [ ]:
# Obtener paths de los fallos de cada modelo
paths_scratch = {f['img_path'] for f in failures_scratch}
paths_effnet  = {f['img_path'] for f in failures_effnet}

shared_paths      = paths_scratch & paths_effnet
only_scratch_paths= paths_scratch - paths_effnet
only_effnet_paths = paths_effnet  - paths_scratch

print(f'Fallos compartidos (ambos modelos):   {len(shared_paths)}')
print(f'Solo TireCNN:                         {len(only_scratch_paths)}')
print(f'Solo EfficientNet-B0:                 {len(only_effnet_paths)}')

# Filtrar listas
shared_failures   = [f for f in failures_effnet if f['img_path'] in shared_paths]
only_eff_failures = [f for f in failures_effnet if f['img_path'] in only_effnet_paths]
only_scr_failures = [f for f in failures_scratch if f['img_path'] in only_scratch_paths]

# Ordenar por confianza del error (más confiados primero = casos más interesantes)
shared_failures.sort(key=lambda x: x['confidence'], reverse=True)
only_eff_failures.sort(key=lambda x: x['confidence'], reverse=True)
only_scr_failures.sort(key=lambda x: x['confidence'], reverse=True)

---
## 4. Visualización detallada de los 5+ casos seleccionados

Seleccionamos los casos más informativos para el informe:
- 3 fallos compartidos (los más difíciles, ambos modelos fallaron)
- 1 fallo exclusivo de TireCNN
- 1 fallo exclusivo de EfficientNet-B0

In [ ]:
def get_gradcam_overlay(model, target_layers, img_tensor, target_class):
    """Devuelve el overlay Grad-CAM para un fallo específico."""
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    rgb  = (img_tensor.squeeze(0).cpu() * std + mean).clamp(0,1)
    rgb  = rgb.permute(1,2,0).numpy().astype(np.float32)

    with GradCAM(model=model, target_layers=target_layers) as cam:
        targets  = [ClassifierOutputTarget(target_class)]
        gray_cam = cam(input_tensor=img_tensor.to(DEVICE), targets=targets)[0]
    return show_cam_on_image(rgb, gray_cam, use_rgb=True), gray_cam


# Capas objetivo
last_conv_scratch = None
for layer in model_scratch.features[-1].block:
    if isinstance(layer, nn.Conv2d):
        last_conv_scratch = layer
target_layers_scratch = [last_conv_scratch]
target_layers_effnet  = [model_effnet.features[-1][0]]


def plot_failure_case(case_number, failure_effnet, failure_scratch,
                      case_type, hypothesis, save_path):
    """
    Panel completo para un caso de fallo:
    - Imagen original
    - Grad-CAM de EfficientNet (activación hacia la clase predicha — incorrecta)
    - Grad-CAM de TireCNN
    - Distribución de probabilidades
    - Metadatos e hipótesis
    """
    has_scratch = failure_scratch is not None
    n_cols = 4 if has_scratch else 3

    fig = plt.figure(figsize=(5 * n_cols, 6))
    gs  = gridspec.GridSpec(2, n_cols, height_ratios=[3, 1], hspace=0.4)

    # ── Imagen original ───────────────────────────────────────────────────────
    ax_img = fig.add_subplot(gs[0, 0])
    ax_img.imshow(failure_effnet['img_pil'].resize((IMG_SIZE, IMG_SIZE)))
    ax_img.set_title('Imagen original', fontweight='bold')
    ax_img.axis('off')

    # Etiqueta con clase real y predicha
    ax_img.text(0.5, -0.08,
                f"Real: {failure_effnet['true_name']}",
                transform=ax_img.transAxes, ha='center',
                fontsize=10, color='#27ae60', fontweight='bold')
    ax_img.text(0.5, -0.16,
                f"Predicho: {failure_effnet['pred_name']} ({failure_effnet['confidence']*100:.1f}%)",
                transform=ax_img.transAxes, ha='center',
                fontsize=10, color='#e74c3c', fontweight='bold')

    # ── Grad-CAM EfficientNet (activando hacia clase predicha = clase errónea) ─
    ax_eff = fig.add_subplot(gs[0, 1])
    cam_eff, _ = get_gradcam_overlay(
        model_effnet, target_layers_effnet,
        failure_effnet['img_tensor'],
        target_class=failure_effnet['pred_label']   # clase errónea
    )
    ax_eff.imshow(cam_eff)
    ax_eff.set_title(f'EfficientNet Grad-CAM\n→ activando clase: "{failure_effnet["pred_name"]}" (ERROR)',
                     fontsize=8, fontweight='bold')
    ax_eff.axis('off')

    # ── Grad-CAM TireCNN ──────────────────────────────────────────────────────
    if has_scratch:
        ax_scr = fig.add_subplot(gs[0, 2])
        cam_scr, _ = get_gradcam_overlay(
            model_scratch, target_layers_scratch,
            failure_scratch['img_tensor'],
            target_class=failure_scratch['pred_label']
        )
        ax_scr.imshow(cam_scr)
        ax_scr.set_title(f'TireCNN Grad-CAM\n→ activando clase: "{failure_scratch["pred_name"]}" (ERROR)',
                         fontsize=8, fontweight='bold')
        ax_scr.axis('off')

    # ── Distribución de probabilidades ───────────────────────────────────────
    ax_prob = fig.add_subplot(gs[0, -1])
    probs   = failure_effnet['probs']
    bars    = ax_prob.barh(CLASS_NAMES, probs,
                           color=['#e74c3c' if i == failure_effnet['pred_label']
                                  else '#bdc3c7' for i in range(NUM_CLASSES)])
    ax_prob.set_xlim(0, 1)
    ax_prob.set_title('Probabilidades\n(EfficientNet)', fontsize=9, fontweight='bold')
    for bar, p in zip(bars, probs):
        ax_prob.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                     f'{p*100:.1f}%', va='center', fontsize=9)
    ax_prob.grid(axis='x', alpha=0.3)
    ax_prob.axvline(0.5, color='gray', linestyle='--', linewidth=1)

    # ── Hipótesis (fila inferior) ─────────────────────────────────────────────
    ax_hyp = fig.add_subplot(gs[1, :])
    ax_hyp.axis('off')

    meta = (
        f"Tipo de error: {failure_effnet['error_type']}  |  "
        f"Brillo: {failure_effnet['brightness']:.1f}  |  "
        f"Contraste: {failure_effnet['contrast']:.1f}  |  "
        f"Nitidez (Lap. var): {failure_effnet['blurriness']:.1f}"
    )
    ax_hyp.text(0.5, 0.75, meta,
                transform=ax_hyp.transAxes, ha='center',
                fontsize=8.5, color='#555', style='italic')
    ax_hyp.text(0.5, 0.35,
                f'💡 Hipótesis: {hypothesis}',
                transform=ax_hyp.transAxes, ha='center',
                fontsize=9.5, color='#2c3e50',
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='#ffeaa7',
                          edgecolor='#fdcb6e', alpha=0.8))

    plt.suptitle(
        f'Caso {case_number} — {case_type}',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Caso {case_number} guardado: {save_path}')


print('✅ Función de visualización de fallos lista.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CASOS 1, 2, 3 — Fallos compartidos (ambos modelos se equivocaron)
# ─────────────────────────────────────────────────────────────────────────────

def get_matching_scratch_failure(path, failures_scratch):
    """Busca el mismo path en los fallos de TireCNN."""
    for f in failures_scratch:
        if f['img_path'] == path:
            return f
    return None


# Hipótesis genéricas que se rellenan automáticamente según el tipo de error
def auto_hypothesis(failure):
    etype = failure['error_type']
    conf  = failure['confidence']
    blur  = failure['blurriness']
    cont  = failure['contrast']

    parts = []
    if 'Falso Negativo' in etype:
        parts.append('El daño es sutil o localizado en una región pequeña de la textura')
    elif 'Falso Positivo' in etype:
        parts.append('La textura presenta variaciones naturales que el modelo confunde con daño')

    if blur < 50:
        parts.append('la imagen tiene baja nitidez (desenfoque o ruido de captura)')
    if cont < 20:
        parts.append('bajo contraste dificulta distinguir surcos de grietas')
    if conf > 0.85:
        parts.append(f'el modelo muestra alta confianza ({conf*100:.0f}%) sugiriendo un sesgo aprendido')
    elif conf < 0.60:
        parts.append(f'baja confianza ({conf*100:.0f}%) indica que el ejemplo está en la frontera de decisión')

    return '. '.join(parts) if parts else 'Caso ambiguo; imagen en frontera de decisión del clasificador.'


# Seleccionar hasta 3 fallos compartidos
n_shared_to_show = min(3, len(shared_failures))
for i, fail_eff in enumerate(shared_failures[:n_shared_to_show]):
    fail_scr = get_matching_scratch_failure(fail_eff['img_path'], failures_scratch)
    plot_failure_case(
        case_number   = i + 1,
        failure_effnet= fail_eff,
        failure_scratch=fail_scr,
        case_type     = f'Fallo compartido (ambos modelos) — {fail_eff["error_type"]}',
        hypothesis    = auto_hypothesis(fail_eff),
        save_path     = f'/content/failure_case_{i+1}.png'
    )

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CASO 4 — Fallo exclusivo de TireCNN (EfficientNet acertó)
# ─────────────────────────────────────────────────────────────────────────────
if only_scr_failures:
    fail_scr = only_scr_failures[0]
    # Obtener la predicción correcta de EfficientNet para la misma imagen
    img_tensor = val_tf(fail_scr['img_pil']).unsqueeze(0)
    with torch.no_grad():
        logits_eff = model_effnet(img_tensor.to(DEVICE))
        probs_eff  = F.softmax(logits_eff, 1).cpu().squeeze().numpy()

    # Construir un pseudo-failure para EfficientNet con la info correcta
    fail_eff_correct = {
        'img_pil':     fail_scr['img_pil'],
        'img_tensor':  img_tensor,
        'true_name':   fail_scr['true_name'],
        'true_label':  fail_scr['true_label'],
        'pred_name':   CLASS_NAMES[probs_eff.argmax()],
        'pred_label':  int(probs_eff.argmax()),
        'confidence':  float(probs_eff.max()),
        'error_type':  '✅ Correcto (EfficientNet)',
        'brightness':  fail_scr['brightness'],
        'contrast':    fail_scr['contrast'],
        'blurriness':  fail_scr['blurriness'],
        'probs':       probs_eff,
    }

    hyp4 = (
        f"TireCNN, con menor capacidad representacional (~1.2M params), "
        f"no logra discriminar este patrón de textura que EfficientNet sí captura "
        f"gracias a sus bloques MBConv con Squeeze-and-Excitation. "
        + auto_hypothesis(fail_scr)
    )

    plot_failure_case(
        case_number   = 4,
        failure_effnet= fail_eff_correct,   # EfficientNet con acierto
        failure_scratch=fail_scr,           # TireCNN con error
        case_type     = 'Fallo exclusivo de TireCNN (EfficientNet acertó)',
        hypothesis    = hyp4,
        save_path     = '/content/failure_case_4.png'
    )
else:
    print('ℹ️  No hay fallos exclusivos de TireCNN — el modelo cometió todos sus errores en imágenes que EfficientNet también erró.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CASO 5 — Fallo exclusivo de EfficientNet (TireCNN acertó)
# ─────────────────────────────────────────────────────────────────────────────
if only_eff_failures:
    fail_eff = only_eff_failures[0]
    hyp5 = (
        f"Paradójicamente, EfficientNet se equivoca en este ejemplo que TireCNN resuelve. "
        f"Posibles causas: sobreajuste del backbone a características de ImageNet "
        f"poco relevantes para texturas industriales, o que el fine-tuning fue insuficiente "
        f"para este tipo de imagen. "
        + auto_hypothesis(fail_eff)
    )
    plot_failure_case(
        case_number   = 5,
        failure_effnet= fail_eff,
        failure_scratch=None,
        case_type     = 'Fallo exclusivo de EfficientNet (TireCNN acertó)',
        hypothesis    = hyp5,
        save_path     = '/content/failure_case_5.png'
    )
else:
    print('ℹ️  No hay fallos exclusivos de EfficientNet.')

---
## 5. Análisis estadístico de los patrones de fallo

In [ ]:
# DataFrame de todos los fallos
all_failures = failures_effnet + failures_scratch
df_fail = pd.DataFrame([{
    'Modelo':        f['model'],
    'Clase real':    f['true_name'],
    'Clase predicha':f['pred_name'],
    'Tipo error':    f['error_type'],
    'Confianza (%)': round(f['confidence'] * 100, 1),
    'Brillo':        round(f['brightness'], 1),
    'Contraste':     round(f['contrast'], 1),
    'Nitidez':       round(f['blurriness'], 1),
} for f in all_failures])

print('📊 Resumen de todos los fallos:')
print(df_fail.groupby(['Modelo', 'Tipo error']).size().reset_index(name='N').to_string(index=False))
df_fail.to_csv('/content/failure_analysis.csv', index=False)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

models_list  = df_fail['Modelo'].unique()
palette      = {'TireCNN': '#e74c3c', 'EfficientNet-B0': '#2980b9'}

# ── 1. Distribución de confianza en los errores ───────────────────────────────
for m in models_list:
    sub = df_fail[df_fail['Modelo'] == m]['Confianza (%)']
    axes[0][0].hist(sub, bins=15, alpha=0.6, color=palette[m], label=m, edgecolor='white')
axes[0][0].axvline(50, color='gray', linestyle='--', linewidth=1, label='Umbral 50%')
axes[0][0].set_title('Confianza del error', fontweight='bold')
axes[0][0].set_xlabel('Confianza (%)'); axes[0][0].set_ylabel('Frecuencia')
axes[0][0].legend(); axes[0][0].grid(alpha=0.3)

# ── 2. Tipo de error por modelo ───────────────────────────────────────────────
ct = df_fail.groupby(['Modelo', 'Tipo error']).size().unstack(fill_value=0)
ct.plot(kind='bar', ax=axes[0][1], color=['#e74c3c', '#f39c12'], edgecolor='white')
axes[0][1].set_title('FP vs FN por modelo', fontweight='bold')
axes[0][1].set_xlabel(''); axes[0][1].tick_params(axis='x', rotation=15)
axes[0][1].legend(fontsize=8); axes[0][1].grid(axis='y', alpha=0.3)

# ── 3. Distribución de brillo en fallos vs total ──────────────────────────────
all_brightness = [ImageStat.Stat(Image.open(raw_dataset.samples[i][0]).convert('RGB')).mean
                  for i in random.sample(test_idx, min(100, len(test_idx)))]
all_brightness = [np.mean(b) for b in all_brightness]
axes[0][2].hist(all_brightness, bins=15, alpha=0.4, color='gray', label='Test completo')
axes[0][2].hist(df_fail[df_fail['Modelo']=='EfficientNet-B0']['Brillo'],
                bins=15, alpha=0.6, color='#2980b9', label='Fallos EffNet')
axes[0][2].set_title('Brillo: test completo vs fallos', fontweight='bold')
axes[0][2].set_xlabel('Brillo medio'); axes[0][2].legend(); axes[0][2].grid(alpha=0.3)

# ── 4. Dispersión confianza vs contraste ─────────────────────────────────────
for m in models_list:
    sub = df_fail[df_fail['Modelo'] == m]
    axes[1][0].scatter(sub['Contraste'], sub['Confianza (%)'],
                       alpha=0.6, color=palette[m], label=m, s=40)
axes[1][0].set_title('Contraste vs Confianza del error', fontweight='bold')
axes[1][0].set_xlabel('Contraste'); axes[1][0].set_ylabel('Confianza (%)')
axes[1][0].axhline(50, color='gray', linestyle='--', linewidth=1)
axes[1][0].legend(); axes[1][0].grid(alpha=0.3)

# ── 5. Dispersión nitidez vs confianza ───────────────────────────────────────
for m in models_list:
    sub = df_fail[df_fail['Modelo'] == m]
    axes[1][1].scatter(sub['Nitidez'], sub['Confianza (%)'],
                       alpha=0.6, color=palette[m], label=m, s=40)
axes[1][1].set_title('Nitidez (Lap. var) vs Confianza', fontweight='bold')
axes[1][1].set_xlabel('Nitidez'); axes[1][1].set_ylabel('Confianza (%)')
axes[1][1].legend(); axes[1][1].grid(alpha=0.3)

# ── 6. Errores por clase real ─────────────────────────────────────────────────
by_class = df_fail.groupby(['Modelo', 'Clase real']).size().unstack(fill_value=0)
by_class.plot(kind='bar', ax=axes[1][2], color=['#27ae60', '#e74c3c'], edgecolor='white')
axes[1][2].set_title('Fallos por clase real', fontweight='bold')
axes[1][2].set_xlabel(''); axes[1][2].tick_params(axis='x', rotation=15)
axes[1][2].legend(fontsize=8); axes[1][2].grid(axis='y', alpha=0.3)

plt.suptitle('Análisis estadístico de patrones de fallo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/failure_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Tabla resumen para el informe

In [ ]:
casos = [
    {
        'Caso': 1, 'Tipo': 'Fallo compartido',
        'Descripción': shared_failures[0]['error_type'] if shared_failures else 'N/A',
        'Hipótesis principal': auto_hypothesis(shared_failures[0]) if shared_failures else 'N/A',
    },
    {
        'Caso': 2, 'Tipo': 'Fallo compartido',
        'Descripción': shared_failures[1]['error_type'] if len(shared_failures) > 1 else 'N/A',
        'Hipótesis principal': auto_hypothesis(shared_failures[1]) if len(shared_failures) > 1 else 'N/A',
    },
    {
        'Caso': 3, 'Tipo': 'Fallo compartido',
        'Descripción': shared_failures[2]['error_type'] if len(shared_failures) > 2 else 'N/A',
        'Hipótesis principal': auto_hypothesis(shared_failures[2]) if len(shared_failures) > 2 else 'N/A',
    },
    {
        'Caso': 4, 'Tipo': 'Exclusivo TireCNN',
        'Descripción': only_scr_failures[0]['error_type'] if only_scr_failures else 'N/A',
        'Hipótesis principal': 'Capacidad representacional insuficiente de TireCNN vs EfficientNet',
    },
    {
        'Caso': 5, 'Tipo': 'Exclusivo EfficientNet',
        'Descripción': only_eff_failures[0]['error_type'] if only_eff_failures else 'N/A',
        'Hipótesis principal': 'Posible sobreajuste a características de ImageNet irrelevantes para texturas industriales',
    },
]

df_casos = pd.DataFrame(casos)
print('\n📋 Tabla resumen de casos de fallo (para el informe):')
print(df_casos.to_string(index=False))
df_casos.to_csv('/content/failure_cases_summary.csv', index=False)

---
## 7. Guardar y descargar

In [ ]:
from google.colab import files

outputs = [
    '/content/failure_case_1.png',
    '/content/failure_case_2.png',
    '/content/failure_case_3.png',
    '/content/failure_case_4.png',
    '/content/failure_case_5.png',
    '/content/failure_statistics.png',
    '/content/failure_analysis.csv',
    '/content/failure_cases_summary.csv',
]
for f_path in outputs:
    if os.path.exists(f_path):
        files.download(f_path)
    else:
        print(f'⚠️  No generado aún: {f_path}')
print('✅ Descargas completadas.')

---
## 📝 Guía de redacción para el informe

### Estructura sugerida para la sección de fallos (≈1 página)

**Párrafo 1 — Panorama general:**  
Reportar cuántos errores cometió cada modelo en el test, desglosados en FP y FN. Destacar que los FN son más críticos en seguridad vial.

**Párrafo 2 — Fallos compartidos (Casos 1-3):**  
Estos representan las instancias más difíciles del dataset. Analizar si comparten características visuales comunes (baja nitidez, iluminación atípica, daño sutil). Apoyarse en los mapas Grad-CAM para mostrar qué región activó el error.

**Párrafo 3 — Fallos exclusivos (Casos 4-5):**  
Caso 4 evidencia las limitaciones de capacidad de TireCNN. Caso 5, si existe, es especialmente relevante porque muestra que el transfer learning no es siempre superior: el backbone puede heredar sesgos de ImageNet.

**Párrafo 4 — Conclusión y mejoras propuestas:**  
- Aumentar datos de llantas con daño sutil (FN más costosos)
- Test-Time Augmentation (TTA) para reducir errores en frontera de decisión
- Umbral de decisión ajustable según el costo relativo de FP vs FN
- Active Learning: etiquetar prioritariamente los ejemplos con baja confianza